In [1]:
import torch
from torch.utils.data import DataLoader, random_split
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import os
import random

from data.preprocessing.pipeline import Pipeline
from data.datasets.MLP_Dataset import MoleculeImageDataset
from models.MLP import ImageOnlyModel
from models.MLP import FocalLoss
from utils.paths import WEIGHTS_DIR

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [3]:
random_state = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 30
batch_size = 64
learning_rate = 1e-3
val_size = 0.2
patience = 5
checkpoint_path="best_model.pth"
save_dir_path = os.path.join(WEIGHTS_DIR, 'MLP')

set_seed(random_state)

In [4]:
pipeline = Pipeline(feature_type="rdkit", download=False)

Extracting raw features:   0%|          | 0/2064 [00:00<?, ?it/s]

Extracting raw features: 100%|██████████| 888/888 [00:04<00:00, 202.84it/s]


In [5]:
import os
import torch
import numpy as np
from torch.utils.data import DataLoader, random_split
from torchvision import transforms

def train_image_model(
    images, labels, epochs=20, batch_size=64, lr=1e-3,
    val_size=0.2, patience=5, checkpoint_path="best_image_model.pth",
    save_dir_path=".", device='cuda'
):
    os.makedirs(save_dir_path, exist_ok=True)

    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.ToTensor(),  # HWC -> CHW, [0,1]
        transforms.Normalize(mean=mean, std=std)
    ])

    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    full_dataset = MoleculeImageDataset(images, labels, transform=None)
    n_total = len(full_dataset)
    n_val = int(n_total * val_size)
    n_train = n_total - n_val
    train_subset, val_subset = random_split(full_dataset, [n_train, n_val])

    for subset in [train_subset, val_subset]:
        subset.dataset.transform = train_transform if subset == train_subset else val_transform

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    model = ImageOnlyModel(num_classes=len(np.unique(labels))).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = FocalLoss(gamma=1.5)

    best_val_loss = float("inf")
    patience_counter = 0
    all_val_preds = []
    all_val_targets = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)

                logits = model(x)
                loss = criterion(logits, y)
                val_losses.append(loss.item())

                preds = torch.argmax(logits, dim=1)
                all_preds.append(preds.cpu().numpy())
                all_targets.append(y.cpu().numpy())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(save_dir_path, checkpoint_path))
            print(f"  → New best model saved! val_loss={best_val_loss:.5f}")
        else:
            patience_counter += 1
            print(f"  (patience {patience_counter}/{patience})")
            if patience_counter >= patience:
                print("\nEarly stopping triggered!")
                break

        all_val_preds = np.concatenate(all_preds)
        all_val_targets = np.concatenate(all_targets)

    macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    print("\nFinal macro F1 on validation:", macro_f1)

    return model, macro_f1, best_val_loss

In [6]:
def predict(model, images, batch_size=64, device='cuda'):
    model.eval()
    dataset = MoleculeImageDataset(images, labels=None)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_preds = []
    with torch.no_grad():
        for x in loader:
            x = x.to(device)
            logits = model(x)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.append(preds)

    return np.concatenate(all_preds)


In [7]:
train_images, test_images = pipeline.get_images()  # train_images: (2064, 244, 244, 3)
train_labels = pipeline.raw_train_data["taste_cluster"].values  # (2064,)

model, macro_f1, best_val_loss = train_image_model(
    images=train_images,
    labels=train_labels,
    epochs=epochs,
    batch_size=batch_size,
    lr=learning_rate,
    val_size=val_size,
    patience=patience,
    checkpoint_path=checkpoint_path,
    save_dir_path=save_dir_path,
    device=device
)

print(f"Training finished. Best validation loss: {best_val_loss:.5f}, macro F1: {macro_f1:.5f}")

Epoch 1/30 | train_loss=12.72380 | val_loss=1.69248
  → New best model saved! val_loss=1.69248
Epoch 2/30 | train_loss=1.58945 | val_loss=1.57817
  → New best model saved! val_loss=1.57817
Epoch 3/30 | train_loss=1.59663 | val_loss=1.53774
  → New best model saved! val_loss=1.53774
Epoch 4/30 | train_loss=1.56700 | val_loss=1.53662
  → New best model saved! val_loss=1.53662
Epoch 5/30 | train_loss=1.54806 | val_loss=1.52495
  → New best model saved! val_loss=1.52495
Epoch 6/30 | train_loss=1.55655 | val_loss=1.54957
  (patience 1/5)
Epoch 7/30 | train_loss=1.56745 | val_loss=1.53637
  (patience 2/5)
Epoch 8/30 | train_loss=1.55501 | val_loss=1.53976
  (patience 3/5)
Epoch 9/30 | train_loss=1.55485 | val_loss=1.56439
  (patience 4/5)
Epoch 10/30 | train_loss=1.54181 | val_loss=1.52597
  (patience 5/5)

Early stopping triggered!

Final macro F1 on validation: 0.051541425818882464
Training finished. Best validation loss: 1.52495, macro F1: 0.05154


In [8]:
model.load_state_dict(torch.load(os.path.join(save_dir_path, checkpoint_path)))
predictions = predict(model, test_images, batch_size=batch_size, device=device)

In [9]:
pred_df = pd.DataFrame({
    "Pubchem_ID": np.arange(len(predictions)),  # если есть реальные ID, подставь сюда
    "taste_cluster": predictions
})

# pred_df.to_csv("test_predictions.csv", index=False)
# print("Predictions saved to test_predictions.csv")

In [10]:
pred_df

,Pubchem_ID,taste_cluster
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1
...,...,...
883,883,1
884,884,1
885,885,1
886,886,1
